# 🎯 XFarmaCare — Segmentación Conductual por Clustering
## Modelo No Supervisado para Perfilar la Cartera

**Objetivo:** Agrupar los 8,000 clientes en segmentos conductuales usando
K-Means sobre variables de comportamiento real: frecuencia de compra,
gasto, canal preferido, adherencia y sentimiento.

Los segmentos resultantes alimentan al Motor de Ofertas y al Índice de Priorización.

**Output:**
- Modelo de clustering exportado (.pkl)
- Scaler del clustering (.pkl)
- CSV con segmento asignado por `customer_id`
---

In [ ]:
# === Librerías ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
import joblib, os, warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
OUTPUT_DIR = "outputs/"
MODEL_DIR = "models/"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# === Cargar el dataset analítico (generado en notebook 01) ===
df = pd.read_csv(f"{OUTPUT_DIR}dataset_analitico_clientes.csv")
print(f"Dataset cargado: {df.shape}")

# Features para clustering: solo las de comportamiento, no las demográficas
# Queremos segmentar por LO QUE HACEN, no por quiénes son
cluster_features = [
    'total_transacciones', 'total_gastado_dop', 'promedio_ticket_dop',
    'frecuencia_mensual', 'recencia_dias', 'productos_distintos',
    'canales_usados', 'descuento_promedio', 'tendencia_gasto',
    'compras_receta_recurrente',
    'total_interacciones', 'ratio_negatividad', 'estrellas_promedio',
    'tasa_adherencia_promedio', 'dias_gap_promedio', 'ratio_retraso'
]

X_cluster = df[cluster_features].fillna(0).copy()
print(f"Features para clustering: {X_cluster.shape[1]}")
X_cluster.describe().round(2)

In [ ]:
# === Escalar los datos ===
scaler_cluster = StandardScaler()
X_scaled = scaler_cluster.fit_transform(X_cluster)

print("Datos escalados correctamente.")
print(f"Media por feature (debe ser ~0): {X_scaled.mean(axis=0).round(2)[:5]}...")
print(f"Std por feature (debe ser ~1):  {X_scaled.std(axis=0).round(2)[:5]}...")

## Método del Codo + Silhouette para elegir K
Buscamos el número óptimo de clusters donde se maximiza la separación
entre grupos sin fragmentar demasiado la cartera.

In [ ]:
# === Método del codo y Silhouette Score ===
rango_k = range(2, 11)
inercias = []
silhouettes = []

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    print(f"K={k}: Inercia={km.inertia_:,.0f} | Silhouette={sil:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rango_k, inercias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo')

axes[1].plot(rango_k, silhouettes, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score por K')

# Marcar el mejor K
mejor_k = list(rango_k)[np.argmax(silhouettes)]
axes[1].axvline(mejor_k, color='green', linestyle='--', label=f'Mejor K={mejor_k}')
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}clustering_elbow_silhouette.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nK óptimo sugerido por Silhouette: {mejor_k}")

In [ ]:
# === Entrenar modelo final con K=5 (o el óptimo) ===
# Usamos K=5 para tener segmentos accionables por el negocio
K_FINAL = 5

modelo_cluster = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20, max_iter=500)
df['cluster'] = modelo_cluster.fit_predict(X_scaled)

sil_final = silhouette_score(X_scaled, df['cluster'])
print(f"Modelo final con K={K_FINAL}")
print(f"Silhouette Score: {sil_final:.4f}")
print(f"\nDistribución de clusters:")
print(df['cluster'].value_counts().sort_index())

In [ ]:
# === Visualización con PCA (2D) ===
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster'],
                     cmap='Set2', alpha=0.5, s=15)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Segmentación de Clientes — Proyección PCA')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}clustering_pca.png", dpi=150, bbox_inches='tight')
plt.show()

## Perfilamiento de Segmentos
Analizamos las características de cada cluster para darle un nombre de negocio.

In [ ]:
# === Perfilar cada cluster ===
perfil = df.groupby('cluster').agg({
    'customer_id': 'count',
    'total_transacciones': 'mean',
    'total_gastado_dop': 'mean',
    'promedio_ticket_dop': 'mean',
    'frecuencia_mensual': 'mean',
    'recencia_dias': 'mean',
    'es_cronico': 'mean',
    'compras_receta_recurrente': 'mean',
    'ratio_negatividad': 'mean',
    'estrellas_promedio': 'mean',
    'tasa_adherencia_promedio': 'mean',
    'descuento_promedio': 'mean',
    'tendencia_gasto': 'mean',
    'programa_lealtad': 'mean'
}).round(2)

perfil.columns = ['n_clientes', 'txn_promedio', 'gasto_total_promedio',
                   'ticket_promedio', 'freq_mensual', 'recencia_dias',
                   'pct_cronicos', 'compras_receta', 'negatividad',
                   'estrellas', 'adherencia', 'descuento_prom',
                   'tendencia_gasto', 'pct_lealtad']

print("PERFIL DE CADA CLUSTER:")
print("="*80)
perfil_t = perfil.T
print(perfil_t.to_string())

In [ ]:
# === Asignar nombres de negocio a cada segmento ===
# Esta lógica se ajusta según el perfil observado arriba

def nombrar_segmento(row, perfil_df):
    """
    Asigna un nombre de negocio basado en las características del cluster.
    Se adapta automáticamente a los datos.
    """
    c = row['cluster']
    p = perfil_df.loc[c]
    
    # Crónico de alto valor: alta adherencia + alto gasto + crónico
    if p['pct_cronicos'] > 0.5 and p['gasto_total_promedio'] > perfil_df['gasto_total_promedio'].median():
        return "Crónico Alto Valor"
    # Crónico en riesgo: crónico pero baja adherencia o alta negatividad
    elif p['pct_cronicos'] > 0.4 and (p['adherencia'] < 0.7 or p['negatividad'] > 0.3):
        return "Crónico en Riesgo"
    # Comprador frecuente: alta frecuencia + buen gasto
    elif p['freq_mensual'] > perfil_df['freq_mensual'].median() * 1.2 and p['pct_cronicos'] <= 0.4:
        return "Frecuente No Crónico"
    # Buscador de ofertas: alto descuento + frecuencia moderada
    elif p['descuento_prom'] > perfil_df['descuento_prom'].median() * 1.3:
        return "Buscador de Ofertas"
    # Esporádico: baja frecuencia + alta recencia
    elif p['recencia_dias'] > perfil_df['recencia_dias'].median() * 1.2:
        return "Esporádico / Inactivo"
    # Dependiente del delivery: alto % de compras no presenciales
    elif p['freq_mensual'] > perfil_df['freq_mensual'].median():
        return "Digital Activo"
    else:
        return "Estándar"

df['segmento'] = df.apply(lambda row: nombrar_segmento(row, perfil), axis=1)

print("\nSegmentos asignados:")
print(df['segmento'].value_counts())

# Visualización
fig, ax = plt.subplots(figsize=(10, 6))
seg_gasto = df.groupby('segmento')['total_gastado_dop'].mean().sort_values(ascending=True)
seg_gasto.plot(kind='barh', ax=ax, color='#264653')
ax.set_title('Gasto Promedio por Segmento')
ax.set_xlabel('Gasto Promedio (DOP)')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}clustering_segmentos_gasto.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Exportar modelo y resultados ===
# Guardar modelo de clustering y scaler
joblib.dump(modelo_cluster, f"{MODEL_DIR}modelo_clustering_xfarmacare.pkl")
joblib.dump(scaler_cluster, f"{MODEL_DIR}scaler_clustering.pkl")
joblib.dump(cluster_features, f"{MODEL_DIR}cluster_features.pkl")

# Exportar segmentos por cliente
output_seg = df[['customer_id', 'cluster', 'segmento']].copy()
output_seg['fecha_segmentacion'] = pd.Timestamp.now().strftime('%Y-%m-%d')
output_seg.to_csv(f"{OUTPUT_DIR}output_segmentos_clientes.csv", index=False)

# Exportar perfil de clusters para referencia
perfil.to_csv(f"{OUTPUT_DIR}perfil_clusters.csv")

print(f"Modelo de clustering guardado en: {MODEL_DIR}")
print(f"Segmentos exportados: {len(output_seg):,} clientes")
print(f"\nArchivos generados:")
print(f"  - modelo_clustering_xfarmacare.pkl")
print(f"  - scaler_clustering.pkl")
print(f"  - output_segmentos_clientes.csv")

## ✅ Segmentación Completada
- **5 clusters** identificados con nombres de negocio accionables
- Modelo y scaler exportados para scoring de nuevos clientes
- Segmentos conectados por `customer_id` al modelo estrella

**Siguiente:** `04_segmentacion_clustering_scoring.ipynb` para clasificar nuevos clientes.